# Учись Машина Учись / Learn Machine Learn

<img src="data/lml.png" width=200>

Онлайн-лекции Ильи С. Елисеева: применение методов машинного обучения в анализе данных.

- Канал в Telegram: https://t.me/learn_machine_learn
- YouTube: https://www.youtube.com/channel/UCCwDwKatNitBCVAJajremMQ
- VK: https://vk.com/learn_machine_learn
- GitHub: https://github.com/easyise/learn_machine_learn

---



## 3. Очистка трека

Какова цель очистки трека?

1. Привести в порядок пространственные показатели: избавиться от телепортаций
2. Привести в порядок скорости и ускорения
3. Выполнить сглаживание трека
4. Первое, второе и третье вместе взятое.

### 3.1 Очистка трека от выбросов

Предагаются следующие подходы:
1. Использовать для очистки трека от нереальных значений фильтрацию по скорости и ускорению, а также триангуляцию.
2. Использовать кластерный анализ для выявления телепортов и удалять соответствующие кластеры.
3. Использовать сглаживающие фильтры


In [ ]:
%pip install gpxpy geopy geopandas folium contextily ipywidgets osmnx

In [ ]:
import pandas as pd
import geopandas as gpd

import numpy as np
import matplotlib.pyplot as plt

import contextily as cx 

%load_ext autoreload
%autoreload 2

from utils.geodata import *
from __config__ import *

In [ ]:
gdf_problem = gpd.read_file('data/Afternoon_Backcountry_Ski.gpx', 
                        layer="track_points")

fields = ['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']

gdf_problem = recalculate_metrics(gdf_problem)
gdf_problem[fields]

In [ ]:
gdf_problem = gdf_problem[gdf_problem['dist'] != 0]  # удалим все сегменты с нулевым расстоянием
gdf_problem[fields].describe()

После каждого удаления выбросов не забываем пересчитать все метрики (расстояния, скорости, ускорения), так как они зависят от последовательности точек маршрута.

In [ ]:
gdf_problem = recalculate_metrics(gdf_problem)
gdf_problem[fields].head(20)

In [ ]:
# Очистка трека от выбросов при помощи z-оценки
def clean_gdf_by_zscore(gdf, z_thresh=6):
    gdf = gdf.copy()
    recalculate_metrics(gdf)
    gdf_metrics_z, v_rob, a_rob = get_z_scores(gdf, z_thresh)
    
    clean_mask = gdf_metrics_z['speed_kmh_z_fail'] 
        # | gdf_metrics_z['acc_m_per_s2_z_fail']
    gdf_clean = gdf_metrics_z[~clean_mask]
    
    return gdf_clean, clean_mask, v_rob, a_rob

gdf_clean, clean_mask, v_rob, a_rob = clean_gdf_by_zscore(gdf_problem, z_thresh=6)
print(f"Robust speed threshold: {v_rob:.2f} km/h")
print("Original points:", len(clean_mask))
print("Cleaned points:", clean_mask.sum())
gdf_problem[['time', 'geometry', 'speed_kmh', 'acc_m_per_s2']][clean_mask]

In [ ]:
# визуализация точек, что мы выбросили
ax = gdf_problem[clean_mask][['time', 'geometry', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=v_rob,
        vmax=80,
        legend=True,
        alpha=0.9,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
recalculate_metrics(gdf_clean)
gdf_clean[fields].describe()



In [ ]:
# визуализация очищенного трека
ax = get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=v_rob,
        legend=True,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
# посчитаем расстояние
print("Расстояние по треку после очистки по выбросам скорости/ускорения {:.2f}".format(gdf_clean.dist.sum()))

In [ ]:
# теперь то же самое, но в цикле, пока в треке есть выбросы
def cleanup_by_zscore_iter(gdf, z_thresh=6, outliers_ratio=0.02, max_iterations=10):
    gdf_clean = gdf.copy()
    for iteration in range(max_iterations):
        gdf_clean, clean_mask, v_rob, a_rob = clean_gdf_by_zscore(gdf_clean, z_thresh)
        n_removed = clean_mask.sum()
        print(f"Iteration {iteration+1}: removed {n_removed} points")
        if n_removed / len(gdf_clean) < outliers_ratio:
            break
    return gdf_clean, v_rob, a_rob

gdf_iter_clean, v_iter_rob, a_iter_rob = cleanup_by_zscore_iter(gdf_problem, z_thresh=6, outliers_ratio=0.02, max_iterations=10)
recalculate_metrics(gdf_iter_clean)
print("Final cleaned track length: {:.2f} m".format(gdf_iter_clean.dist.sum()))
print("Final robust speed threshold: {:.2f} km/h".format(v_iter_rob))
gdf_iter_clean[['dist', 'speed_kmh', 'acc_m_per_s2']].describe()

In [ ]:
ax = get_with_segments(gdf_iter_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=v_rob,
        legend=True,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
gdf_problem = gdf_iter_clean
gdf_problem

In [ ]:
# посмотрим на качество триангуляции
recalculate_triangulation_metrics(gdf_problem)
gdf_problem[['speed_kmh', 'speed_out', 'speed_base', 'triang_diff']].describe()

In [ ]:
gdf_problem_ts = gdf_problem.set_index('time').sort_index()

fig, ax = plt.subplots(figsize=(12, 4))

# gdf_ts_disp = gdf_ts['2026-01-18 13:26':'2026-01-18 13:38']
gdf_problem_ts['triang_diff'].plot(ax=ax, title='Triangulation Difference')

fig.tight_layout()
plt.show()

In [ ]:
triang_thres = 0.8
threshold_speed = v_iter_rob
mask_triang = ((gdf_problem_ts['triang_diff'] < triang_thres) & 
               ( (gdf_problem_ts['speed_kmh'] > threshold_speed) |
                   (gdf_problem_ts['speed_out'] > threshold_speed)
                 )
               )
print(f"Выбрасываем {(mask_triang).sum()} точек с плохой триангуляцией")
gdf_problem_ts_clean = gdf_problem_ts[~mask_triang]
gdf_problem_ts_clean = recalculate_metrics(gdf_problem_ts_clean.reset_index())
print("Расстояние по треку после очистки по триангуляции {:.2f} м".format(gdf_problem_ts_clean.dist.sum())) 
ax = get_with_segments(gdf_problem_ts_clean)[['segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=v_iter_rob,
        legend=True,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
gdf_problem = gpd.read_file('data/Afternoon_Backcountry_Ski.gpx', 
                        layer="track_points")

fields = ['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']

gdf_problem = recalculate_metrics(gdf_problem)
gdf_problem = gdf_problem[gdf_problem['dist'] != 0]  # удалим все сегменты с нулевым расстоянием
gdf_problem, v_rob, a_rob = get_z_scores(gdf_problem, z=6) # получим значение робастных порогов
gdf_problem[fields + ['speed_kmh_z', 'acc_m_per_s2_z']].describe()

In [ ]:
# Запускаем очистку в цикле, пока есть выбросы по триангуляции
def cleanup_triang_iter(gdf_, outliers_ratio=0.0, triang_thres=0.8, speed_thresh=30):
    gdf_ = gdf_.copy()
    iteration = 0
    num_outliers_target = int(gdf_.shape[0] * outliers_ratio)
    while True:
        recalculate_triangulation_metrics(gdf_)
        mask_triang = (((gdf_['triang_diff'] < triang_thres) | (gdf_['speed_base'] > speed_thresh )) 
                       & ( (gdf_['speed_kmh'] > speed_thresh) |
                         (gdf_['speed_out'] > speed_thresh)
                       )
                     )
        num_outliers = mask_triang.sum()
        if num_outliers <= num_outliers_target:
            print(f"Итерация {iteration}: не выбрасываем {num_outliers} точек (<= {num_outliers_target}), конец")
            break
        print(f"Итерация {iteration}: выбрасываем {num_outliers} точек с плохой триангуляцией (> {num_outliers_target})")
        gdf_ = gdf_[~mask_triang]
        gdf_ = recalculate_metrics(gdf_)
        iteration += 1
    return gdf_

gdf_clean = cleanup_triang_iter(gdf_problem, outliers_ratio=0.001, triang_thres=0.8, speed_thresh=v_rob)
gdf_clean[['dist', 'speed_kmh', 'acc_m_per_s2', 'speed_kmh', 'speed_out', 'speed_base']].describe()

In [ ]:
print("Расстояние по треку после очистки по триангуляции {:.2f} м".format(gdf_clean.dist.sum()))

ax = get_with_segments(gdf_clean)[['segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        legend=True,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
gdf_problem = gpd.read_file('data/Bike_bad.gpx', 
                        layer="track_points")

fields = ['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']

gdf_problem = recalculate_metrics(gdf_problem)
gdf_problem = gdf_problem[gdf_problem['dist'] != 0]  # удалим все сегменты с нулевым расстоянием
gdf_problem, v_rob, a_rob = get_z_scores(gdf_problem, z=6) # получим значение робастных порогов
gdf_problem[fields + ['speed_kmh_z', 'acc_m_per_s2_z']].describe()

In [ ]:
ax = get_with_segments(gdf_problem)[['segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        legend=True,
        vmin=0,
        vmax=v_rob,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
gdf_problem = cleanup_by_zscore_iter(gdf_problem, z_thresh=6, outliers_ratio=0.02, max_iterations=10)[0]
recalculate_metrics(gdf_problem)
gdf_clean = cleanup_triang_iter(gdf_problem, outliers_ratio=0.02, triang_thres=0.8, speed_thresh=v_rob)
recalculate_metrics(gdf_clean)
display(gdf_clean[['dist', 'speed_kmh', 'acc_m_per_s2', 'speed_kmh', 'speed_out', 'speed_base']].describe())

ax = get_with_segments(gdf_clean)[['segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        legend=True,
        vmin=0,
        vmax=v_rob,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

### 3.2 Очистка с помощью кластерного анализа DBSCAN

Теперь наша задача - определить, какие из кластеров являются "хорошими", а какие - "плохими" (выбросами).

Для этого можно использовать различные эвристики, например:
 - Средняя скорость в кластере должна быть в разумных пределах (например, не выше 50 км/ч для велосипедиста).
 - Среднее ускорение в кластере должно быть в разумных пределах (например, не выше 5 м/с²).
 - Продолжительность кластера должна быть достаточно длинной (например, не менее 1 минуты).
 - Применение триангуляции: прибытие в кластер и отъезд из него должны быть по скорости сопоставимы с передвижением от предыдущего к следующему кластеру

In [ ]:
gdf_bike_bad = gpd.read_file('data/Bike_bad.gpx', 
                        layer="track_points")

gdf_bike_bad = recalculate_metrics(gdf_bike_bad)
gdf_bike_bad

In [ ]:
labels, eps = get_cluster_labels(gdf_bike_bad, min_samples=2*60)
gdf_bike_bad["cluster_full_label"] = labels
gdf_bike_bad["cluster_full_label"] = gdf_bike_bad["cluster_full_label"].astype("category")
n_clusters = np.unique(labels[labels != -1]).size
noise_points = (labels == -1).sum()
print(f"Найдено кластеров: {n_clusters}, шумовых точек: {noise_points}, eps: {eps:.2f}")

ax = gdf_bike_bad[['time', 'geometry', 'cluster_full_label']] \
    .to_crs(epsg=3857) \
    .plot(
        column="cluster_full_label",
        cmap="tab10",
        legend=True,
        alpha=1,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("Clustering labels: {} clusters".format(n_clusters))
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1)

ax.grid()
plt.show()

In [ ]:
# упорядочим кластеры по времени появления
gdf_bike_bad.groupby('cluster_full_label')['time'].first().sort_values()

In [ ]:
# отдельно рассмотрим кластер -1 (шум)
gdf_noise = gdf_bike_bad[gdf_bike_bad['cluster_full_label'] == -1].copy()
recalculate_metrics(gdf_noise)
print("Продолжительность, с: ", gdf_noise.timedelta_s.sum())
print("Пройденное расстояние, м: ", gdf_noise.dist.sum())
gdf_noise[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'geometry']].describe()

In [ ]:
# 1. напишем функцию, которая дает нам геопанду с данными по кластерам: время и координаты входа в кластер и выхода из него
def get_gdf_clusters_info(gdf_clustered: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    def calc_distance(group):
        if len(group) < 2:
            return np.nan
        crs = group.estimate_utm_crs()
        group_ = group.to_crs(crs)
        return group_.distance(group_.shift(1)).sum()

    gdf_cluster_info = (
        gdf_clustered[['time', 'geometry']]
        .groupby(gdf_clustered['cluster_full_label'])
        .agg(
            time_entry=('time', 'first'),
            time_exit=('time', 'last'),
            point_entry=('geometry', 'first'),
            point_exit=('geometry', 'last'),
            distance=('geometry', calc_distance),
        )
        .sort_values(by='time_entry')
    )
    gdf_cluster_info['speed_kmh'] = gdf_cluster_info['distance'] / gdf_cluster_info['time_exit'].subtract(gdf_cluster_info['time_entry']).dt.total_seconds() * 3.6
    return gdf_cluster_info

gdf_cluster_info = get_gdf_clusters_info(gdf_bike_bad)
gdf_cluster_info
    



In [ ]:
# напишем функцию для расчетов метрик кластеров: скорость входа/выхода/базы, расстояния и времени между кластерами
def recalculate_cluster_metrics(gdf_cluster_info: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # упорядочиваем по времени входа в кластер
    gdf_cluster_info.sort_values(by='time_entry', inplace=True)

    # добавим колонки с данными о покидании предыдущего кластера
    gdf_cluster_info['time_exit_prev'] = gdf_cluster_info['time_exit'].shift(1)
    gdf_cluster_info['point_exit_prev'] = gdf_cluster_info['point_exit'].shift(1)
    # добавим информацию о точках следующего кластера
    gdf_cluster_info['time_entry_next'] = gdf_cluster_info['time_entry'].shift(-1)
    gdf_cluster_info['point_entry_next'] = gdf_cluster_info['point_entry'].shift(-1)
    
    # посчитаем расстояние и время между кластерами
    gdf_cluster_info['dist_in'] = gdf_cluster_info.apply(lambda row: haversine(
        row.point_exit_prev.y if pd.notnull(row.point_exit_prev) else row.point_entry.y, 
        row.point_exit_prev.x if pd.notnull(row.point_exit_prev) else row.point_entry.x,
        row.point_entry.y, row.point_entry.x
    ), axis=1)
    gdf_cluster_info['time_in'] = (gdf_cluster_info['time_entry'] - gdf_cluster_info['time_exit_prev']).dt.total_seconds()
    gdf_cluster_info['speed_in'] = (gdf_cluster_info['dist_in'] / gdf_cluster_info['time_in'] * 3.6).fillna(0)   
    
    # посчитаем расстояние, время и скорость "на выход" и "по базе"
    gdf_cluster_info['dist_out'] = gdf_cluster_info['dist_in'].shift(-1)
    gdf_cluster_info['time_out'] = gdf_cluster_info['time_in'].shift(-1)
    gdf_cluster_info['speed_out'] = gdf_cluster_info['speed_in'].shift(-1)   
    
    gdf_cluster_info['dist_base'] = gdf_cluster_info.apply(lambda row: haversine(
        row.point_exit_prev.y if pd.notnull(row.point_exit_prev) else row.point_entry.y, 
        row.point_exit_prev.x if pd.notnull(row.point_exit_prev) else row.point_entry.x,
        row.point_entry_next.y if pd.notnull(row.point_entry_next) else row.point_entry.y, 
        row.point_entry_next.x if pd.notnull(row.point_entry_next) else row.point_entry.x
    ), axis=1)
    gdf_cluster_info['time_base'] = (gdf_cluster_info['time_entry_next'] - gdf_cluster_info['time_exit_prev']).dt.total_seconds()
    gdf_cluster_info['speed_base'] = (gdf_cluster_info['dist_base'] / gdf_cluster_info['time_base'] * 3.6)
    
    return gdf_cluster_info


gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)
gdf_cluster_info

In [ ]:
# напишем функцию, которая возвращает список кластеров для удаления по заданным критериям
def identify_bad_clusters(gdf_clustered: gpd.GeoDataFrame,
                            v_lim=70) -> list:
    gdf_cluster_info = get_gdf_clusters_info(gdf_clustered)
    gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)

    bad_clusters = []
    reasons = {}

    # 1. шум (-1) - проверяем, если он пересекается по времени с кластерами, удаляем
    noise_mask = gdf_cluster_info.index == -1
    noise_entry = gdf_cluster_info[noise_mask]['time_entry'].min()
    noise_exit = gdf_cluster_info[noise_mask]['time_exit'].max()
    overlapping_clusters = gdf_cluster_info[~noise_mask][
        ( (gdf_cluster_info['time_entry'] <= noise_exit) &
          (gdf_cluster_info['time_exit'] >= noise_entry) ) 
    ].index.tolist()
    if overlapping_clusters:
        gdf_cluster_info.drop(index=[-1], inplace=True)
        bad_clusters.extend([-1])
        reasons = { -1: "overlapping noise cluster" }

    # 2. удаляем вложенные по времени кластеры итеративно, пока таковых не останется
    while True:
        gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)
        time_in = (gdf_cluster_info['time_entry'] - gdf_cluster_info['time_exit_prev']).dt.total_seconds()
        if gdf_cluster_info[time_in < 0].empty:
            break
        nested_cluster = gdf_cluster_info[time_in < 0].index[0]
        bad_clusters.extend([int(nested_cluster)])
        reasons[int(nested_cluster)] = "nested cluster"
        gdf_cluster_info.drop(index=[nested_cluster], inplace=True)
    

    # 3. вычислим метрики входа-выхода-базы для каждого кластера
    while True:
        gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)
        # вычислим коэффициент триангуляции
        max_speed = gdf_cluster_info[['speed_in', 'speed_out']].max(axis=1)
        gdf_cluster_info['triang_diff'] = (gdf_cluster_info['speed_base'] / max_speed).fillna(1)
        triang_diff_mask = (
            ( (gdf_cluster_info['speed_in'] > v_lim) &
              (gdf_cluster_info['speed_out'] > v_lim) ) &
            (gdf_cluster_info['triang_diff'] < 0.8)
        )
        triang_bad_clusters = gdf_cluster_info[triang_diff_mask].index.tolist()
        # display(gdf_cluster_info[['speed_in', 'speed_out', 'speed_base', 'triang_diff']])
        if not triang_bad_clusters:
            break
        to_remove = triang_bad_clusters[0]
        gdf_cluster_info.drop(index=[to_remove], inplace=True)
        bad_clusters.extend([to_remove])         
        reasons[int(to_remove)] = "triangulation difference"
    return bad_clusters, reasons  # уникальные значения

bad_clusters, reasons = identify_bad_clusters(gdf_bike_bad)
print(bad_clusters)
display(reasons)

In [ ]:
# отобразим плохие кластеры на карте
gdf_bike_bad['suspicious_clusters'] = gdf_bike_bad['cluster_full_label'].isin(bad_clusters)


ax = gdf_bike_bad[['time', 'geometry', 'suspicious_clusters']] \
    .to_crs(epsg=3857) \
    .plot(
        column="suspicious_clusters",
        cmap="tab10",
        legend=True,
        alpha=0.9,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("Suspicious cluster labels: {}".format(sorted(bad_clusters)))
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1)

ax.grid()
plt.show()

In [ ]:
# собственно, очистка
gdf_bike_clean = gdf_bike_bad[~gdf_bike_bad['suspicious_clusters']].copy()

gdf_bike_clean = recalculate_metrics(gdf_bike_clean)

gdf_bike_clean[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'geometry']].describe()




In [ ]:
gdf_bike_clean_z, v_rob, a_rob = get_z_scores(gdf_bike_clean, 6)
gdf_bike_clean_z[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'geometry', 'speed_kmh_z', 'acc_m_per_s2_z']].describe()

In [ ]:
gdf_bike_clean_z['speed_kmh_z_fail'].sum()

In [ ]:
gdf_bike_clean, v_iter_rob, a_iter_rob = cleanup_by_zscore_iter(gdf_bike_clean, z_thresh=6, outliers_ratio=0.02, max_iterations=10)
recalculate_metrics(gdf_iter_clean)
gdf_bike_clean[['dist', 'speed_kmh', 'acc_m_per_s2']].describe()

In [ ]:
ax = get_with_segments(gdf_bike_clean)[['segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        legend=True,
        vmin=0,
        vmax=80,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
# снова выполним кластеризацию
labels, eps = get_cluster_labels(gdf_bike_clean, min_samples=2*60)
gdf_bike_clean["cluster_full_label"] = labels
gdf_bike_clean["cluster_full_label"] = gdf_bike_clean["cluster_full_label"].astype("category")
n_clusters = np.unique(labels[labels != -1]).size
noise_points = (labels == -1).sum()
print(f"Найдено кластеров: {n_clusters}, шумовых точек: {noise_points}, eps: {eps:.2f}")


In [ ]:
ax = gdf_bike_clean[['time', 'geometry', 'cluster_full_label']] \
    .to_crs(epsg=3857) \
    .plot(
        column="cluster_full_label",
        cmap="turbo",
        legend=True,
        alpha=0.9,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by cluster labels")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.grid()
plt.show()

### 3.3 Заполнение пропусков в треке



In [ ]:
gdf_problem = gpd.read_file('data/Afternoon_Backcountry_Ski.gpx', 
                        layer="track_points")

fields = ['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']

gdf_problem = recalculate_metrics(gdf_problem)
gdf_problem[fields]

In [ ]:
# выполним кластеризацию
labels, eps = get_cluster_labels(gdf_problem, min_samples=2*60)
gdf_problem["cluster_full_label"] = labels
gdf_problem["cluster_full_label"] = gdf_problem["cluster_full_label"].astype("category")
n_clusters = np.unique(labels[labels != -1]).size

print(f"Найдено кластеров: {n_clusters}, шумовых точек: {(labels == -1).sum()}, eps: {eps:.2f}")

# определим плохие кластеры и удалим их
bad_clusters, reasons = identify_bad_clusters(gdf_problem)
print("Плохие кластеры для удаления:", bad_clusters)
print("Причины удаления кластеров:", reasons)

gdf_clean = gdf_problem[~gdf_problem['cluster_full_label'].isin(bad_clusters)].copy()
gdf_clean = recalculate_metrics(gdf_clean)
print("Расстояние по треку после очистки по кластерам {:.2f} м".format(gdf_clean.dist.sum()))

In [ ]:
# определим разрывы в треке
gdf_clean[['timedelta_s', 'dist', 'speed_kmh']].describe()

gdf_clean, dist_rob, time_rob = get_z_scores(gdf_clean, z=6, cols=['dist', 'timedelta_s'])
print(f"Робастный порог по расстоянию: {dist_rob:.2f} m, робастный порог по времени: {time_rob:.2f} s")

In [ ]:
print("Всего разрывов по расстоянию:", gdf_clean['dist_z_fail'].sum(), 
      "по времени:", gdf_clean['timedelta_s_z_fail'].sum())

In [ ]:
by = 'timedelta_s'
# by = 'dist'

gdf_clean.sort_values(by=by, ascending=False)[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'geometry', 'prev_point', by+'_z', by+'_z_fail']].head(20)

In [ ]:
# визуализируем разрывы на карте
ax = gdf_clean[gdf_clean[by+'_z_fail']][['time', 'geometry', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(figsize=(10, 10),color='red', alpha=0.9, markersize=20)
gdf_clean.shift(1)[gdf_clean[by+'_z_fail']][['time', 'geometry', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(color='blue', alpha=0.9, markersize=20, ax=ax)

get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=80,
        legend=True,
        alpha=0.5,        linewidth=5,
        figsize=(10, 10),
        ax=ax
)


ax.set_title(f"GPX track, cleaned")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
import osmnx as ox

def graph_for_two_points_bbox(lat1, lon1, lat2, lon2, buffer_m=400, network_type="walk"):
    # bbox: north, south, east, west
    buffer_deg = buffer_m / 111320  # приблизительное преобразование метров в градусы
    north = max(lat1, lat2) + buffer_deg
    south = min(lat1, lat2) - buffer_deg
    east  = max(lon1, lon2) + buffer_deg
    west  = min(lon1, lon2) - buffer_deg

    # добавим буфер (в метрах) — osmnx сам корректно расширит bbox
    G = ox.graph_from_bbox(
        (north, south, east, west),
        network_type=network_type,
        simplify=True
    )
    return G

# пример для двух точек
gdf_gap = gdf_clean.sort_values(by=by, ascending=False).iloc[0]
p1 = gdf_gap['prev_point']
p2 = gdf_gap['geometry']

G = graph_for_two_points_bbox(p1.y, p1.x, p2.y, p2.x, buffer_m=400, network_type="all")
G

In [ ]:
# API_KEY = "123456789abcdef123456789abcdef123456789abcdef123456789abcdef123456789"

import requests

from shapely.geometry import LineString

start = (p1.x, p1.y)  # lon, lat
end   = (p2.x, p2.y)  # lon, lat

url = "https://api.openrouteservice.org/v2/directions/foot-walking/geojson"

body = {
    "coordinates": [start, end],
    "elevation": True,
}

headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

r = requests.post(url, json=body, headers=headers)
data = r.json()
data


In [ ]:
data['features'][0]['geometry']['coordinates']

In [ ]:
coords = data['features'][0]['geometry']['coordinates']

gdf_route = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(
        [c[0] for c in coords],
        [c[1] for c in coords],
    ),
    crs="EPSG:4326"
)
gdf_route['ele'] = [c[2] for c in coords]
gdf_route

In [ ]:
# визуализируем разрывы на карте
ax = get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=80,
        legend=True,
        alpha=0.5,        linewidth=5,
        figsize=(10, 10),
)

get_with_segments(gdf_route)\
    .to_crs(epsg=3857) \
    .plot(
        color='red',
        alpha=0.7, linewidth=5,
        ax=ax
)


ax.set_title(f"GPX track + route from API")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
def attach_time_ele_constant_speed(
    gdf: gpd.GeoDataFrame,
    t_end,
    t_delta,
):
    """
    Назначает время и высоту точкам маршрута при постоянной скорости.

    Параметры:
    ----------
    gdf : GeoDataFrame с Point геометрией (в порядке маршрута)
    t_end : datetime-like
    t_delta : timedelta-like

    ele_start, ele_end : высоты (м)

    Возвращает:
    ----------
    GeoDataFrame с колонками:
    seq, seg_dist_m, cum_dist_m, time, ele, speed_mps
    """
    out = gdf.copy()

    t_start = pd.Timestamp(t_end) - t_delta
    t_end = pd.Timestamp(t_end)

    total_sec = t_delta.total_seconds()
    if total_sec <= 0:
        raise ValueError("t_end должен быть позже t_start")

    # --- автоматический UTM ---
    utm_crs = out.estimate_utm_crs()
    out_m = out.to_crs(utm_crs)

    # --- расстояния ---
    seg = out_m.geometry.distance(out_m.geometry.shift(1)).fillna(0.0)
    cum = seg.cumsum()

    total_dist = float(cum.iloc[-1])
    if total_dist == 0:
        raise ValueError("Нулевая длина маршрута")

    # --- постоянная скорость ---
    speed_mps = total_dist / total_sec

    # --- время ---
    seconds = cum / total_dist * total_sec
    times = t_start + pd.to_timedelta(seconds, unit="s")

    # --- запись ---
    out["seq"] = range(len(out))
    out["seg_dist_m"] = seg.values
    out["cum_dist_m"] = cum.values
    out["time"] = times
    out["speed_mps"] = speed_mps

    out.attrs["utm_crs"] = utm_crs

    return out

gdf_route_w_time = attach_time_ele_constant_speed(
    gdf_route,
    t_end=gdf_gap['time'],
    t_delta=pd.Timedelta(seconds=gdf_gap['timedelta_s']),
)

gdf_route_w_time

In [ ]:
gdf_clean['time_prev'] = gdf_clean['time'].shift(1)
gdf_clean.sort_values(by='timedelta_s', ascending=False)[['time', 'time_prev', 'timedelta_s', 'dist']].head(5)

In [ ]:
gdf_route_w_time.iloc[1:-2]

In [ ]:
# объединим трек с маршрутом
gdf_combined = pd.concat([gdf_clean, gdf_route_w_time.iloc[1:-2]], ignore_index=True)
gdf_combined = gdf_combined.sort_values(by='time').reset_index(drop=True)
gdf_combined = recalculate_metrics(gdf_combined)
gdf_combined[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'geometry']].describe()

In [ ]:
ax = gdf_combined[['time', 'geometry', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot( 
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=80,
        legend=True,
        alpha=0.9,
        markersize=5,
        figsize=(10, 10),
)

ax.set_title("Combined GPX track + API route colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
ax = get_with_segments(gdf_combined)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        vmin=0,
        vmax=60,
        legend=True,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)
ax.set_title("Combined GPX track + API route, colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show();

In [ ]:
# посчитаем расстояние по треку после очистки и восстановления разрыва
print("Расстояние по треку после очистки и восстановления разрыва {:.2f} м".format(gdf_combined.dist.sum()))

In [ ]:
# теперь выгрузим результат в GPX
from lxml import etree
def gdf_to_gpx(gdf, filename):
    gpx = etree.Element("gpx", version="1.1", creator="GPX Generator")
    trk = etree.SubElement(gpx, "trk")
    trkseg = etree.SubElement(trk, "trkseg")

    for _, row in gdf.iterrows():
        trkpt = etree.SubElement(trkseg, "trkpt", lat=str(row.geometry.y), lon=str(row.geometry.x))
        time_elem = etree.SubElement(trkpt, "time")
        time_elem.text = row.time.isoformat() + "Z"
        ele_elem = etree.SubElement(trkpt, "ele")
        ele_elem.text = str(row.ele)

    tree = etree.ElementTree(gpx)
    tree.write(filename, pretty_print=True, xml_declaration=True, encoding="UTF-8")

In [ ]:
gdf_problem = gdfs_[3].copy()  # Bike_bad.gpx

fields = ['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']

gdf_problem = recalculate_metrics(gdf_problem)
gdf_problem = gdf_problem[gdf_problem['dist'] != 0]  # удалим все сегменты с нулевым расстоянием
gdf_problem, v_rob, a_rob = get_z_scores(gdf_problem, z=6) # получим значение робастных порогов



labels, eps_m = get_cluster_labels(gdf_problem, min_samples=120)
print(f"Выбранный eps: {eps_m:.2f} м")
gdf_problem['dbscan_label'] = labels
gdf_problem['dbscan_label'] = gdf_problem['dbscan_label'].astype('category')

ax = gdf_problem[['time', 'geometry', 'dbscan_label']] \
    .to_crs(epsg=3857) \
    .plot(
        column="dbscan_label",
        cmap="tab10",
        legend=True,
        alpha=0.9,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by DBSCAN labels")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1)

ax.grid()
plt.show()

In [ ]:
gdf_problem[fields+['dbscan_label']].groupby('dbscan_label').agg(['first', 'last', 'count', 'mean']).sort_values(by=('time', 'first'), ascending=True)

In [ ]:
gdf_problem.groupby('dbscan_label')['time'].first().sort_values()

In [ ]:
clusters = []

for cluster_label, cluster_data in gdf_problem.groupby('dbscan_label')['time'].first().sort_values().items():
    print(f"Cluster {cluster_label} starts at a time: {cluster_data}")
    # в каждом кластере выполняем очистку от выбросов отдельно по z-оценке и триангуляции
    gdf_cluster = gdf_problem[gdf_problem['dbscan_label'] == cluster_label]
    gdf_cluster_clean, v_cluster_rob, a_cluster_rob = cleanup_by_zscore_iter(gdf_cluster, z_thresh=6, outliers_ratio=0.02, max_iterations=10)
    gdf_cluster_clean = cleanup_triang_iter(gdf_cluster_clean, outliers_ratio=0.02, triang_thres=0.8, speed_thresh=v_cluster_rob)
    clusters.append(gdf_cluster_clean)

clusters
